In [ ]:
# from vllm import LLM, SamplingParams
# model_name = "Qwen/Qwen2-VL-2B-Instruct"
# llm = LLM(model=model_name)

In [ ]:
# import sys
# sys.path.append("data/g3/raw")
# from dataset import MsgPackIterableDatasetMultiTargetWithDynLabels

import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import io
import base64
from datasets import Image

def list_to_dict(list_of_dicts):
    result = {}
    for d in list_of_dicts:
        for key, value in d.items():
            result.setdefault(key, []).append(value)
    return result

In [ ]:
# image, problem, solution
import json
target_mapping = json.load(open("data/g3/raw/train/countries.json"))
ds = iter(MsgPackIterableDatasetMultiTargetWithDynLabels(
    path="data/g3/raw/train/msgpack",
    target_mapping=target_mapping,
    key_img_id="id",
    key_img_encoded="image",
    shuffle=False,
    cache_size=350000,
))

In [ ]:


problem = "Based on visual cues, such as road signs, architecture, vegetation, and landscape, identify the most likely country where this image was taken."
country_to_id = pd.read_csv("data/g3/raw/countries.csv", skiprows=0)
country_to_id = {c["class_label"]: c["country"] for c in country_to_id.to_dict(orient="records")}
hf = []
counter = 0
for x in tqdm(ds):
    ann = {"image": x[0], "id": x[2], "solution": f"<answer>{country_to_id[x[1]]}</answer>", "problem": problem}
    hf.append(ann)
    counter += 1
    # if counter > 10:
    #     break

In [ ]:
import numpy as np
idxs = np.random.permutation(range(len(hf)))

In [ ]:
# chunk = [hf[i] for i in idxs[:10000]]
chunk = hf
chunk = list_to_dict(chunk)
chunk = Dataset.from_dict(chunk)
chunk = chunk.cast_column("image", Image())
chunk.to_parquet(f"data/g3/data/val.parquet")

In [ ]:
chunk_size = 25000
n_chunks = (len(hf) + chunk_size - 1) // chunk_size  # Calculate number of chunks

for i in tqdm(range(n_chunks)):
    start = i * chunk_size
    end = min(start + chunk_size, len(hf))
    chunk = hf[start:end]
    chunk = list_to_dict(chunk)
    chunk = Dataset.from_dict(chunk)
    chunk = chunk.cast_column("image", Image())
    chunk.to_parquet(f"data/g3/g3-streetview-300k/data/train_{str(i).zfill(3)}.parquet")

In [5]:
import datasets
ds = datasets.load_dataset(f"data/g3-streetview-9k-balanced/data")
len(ds["train"])
# ds = ds.remove_columns(["image"])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 9000 examples [00:00, 26581.73 examples/s]
Generating validation split: 3887 examples [00:00, 23771.64 examples/s]


9000

In [ ]:
chunk = samples
chunk = list_to_dict(chunk)
chunk = Dataset.from_dict(chunk)
chunk = chunk.cast_column("image", Image())
chunk.to_parquet(f"data/g3/g3-streetview-9k-balanced/train.parquet")

In [ ]:
samples[1]

In [ ]:
for ann in samples:
    ann["image"] = ds["train"][ann["idx"]]["image"]

In [ ]:
from tqdm import tqdm

country_to_id = {}
for i, ann in tqdm(enumerate(ds["train"])):
    ann["idx"] = i
    country_to_id[ann["solution"]] = country_to_id.get(ann["solution"], []) + [ann]

In [ ]:
import numpy as np
country_to_id = {k: np.random.permutation(v)[:100] for k, v in country_to_id.items()}

In [ ]:
samples = []
for k, v in country_to_id.items():
  if "San Marino" not in k and "Andorra" not in k:
    samples.extend(v.tolist())

In [ ]:
sorted_counts = dict(sorted({k: len(v) for k, v in country_to_id.items()}.items(), key=lambda item: item[1]))
sorted_counts

In [ ]:
# 90 * 100